# Backtest Engine Walkthrough

This notebook turns the Week 5 backtest project into a clean notebook format.
The goal is to keep the code runnable first, then explain each part step by step.

## Learning Goals

- load historical price data from a CSV file
- calculate a Simple Moving Average (SMA)
- generate BUY, SELL, HOLD, and warm-up signals
- simulate trades with a simple strategy
- measure return, trade count, and win rate

## Strategy Rules

- `BUY` when `Close > SMA(period)`
- `SELL` when `Close < SMA(period)`
- `HOLD` when `Close == SMA(period)`
- `No signal yet` while the SMA window is still warming up

This is a learning backtest, so it keeps the rules simple: one position at a time and all-in/all-out trades.

In [1]:
import csv
from pathlib import Path

## 1. Define The Simulator Class

The full class stays in one code cell so the notebook can run top to bottom without breaking.

In [2]:
class StrategySimulator:
    def __init__(self):
        self.data = []
        self.trade = []
        self.closes = []
        self.starting_cash = 10000
        self.final_cash = 10000

    def load(self, path):
        self.data = []
        self.closes = []
        path = Path(path)

        with path.open("r", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            self.data = list(reader)

        for row in self.data:
            self.closes.append(float(row["Close"]))

        return len(self.closes)

    def sma(self, period):
        result = []
        for i in range(len(self.closes)):
            if i < period - 1:
                result.append(None)
            else:
                window = self.closes[i - period + 1 : i + 1]
                result.append(sum(window) / period)
        return result

    def generate_signals(self, period):
        result = []
        sma_list = self.sma(period)

        for i in range(len(self.closes)):
            if sma_list[i] is None:
                result.append("No signal yet")
            elif self.closes[i] > sma_list[i]:
                result.append("BUY")
            elif self.closes[i] < sma_list[i]:
                result.append("SELL")
            else:
                result.append("HOLD")

        return result

    def backtest(self, period):
        signals = self.generate_signals(period)
        self.trade = []

        cash = self.starting_cash
        position = 0
        btc_balance = 0

        for i in range(len(self.closes)):
            price = self.closes[i]

            if signals[i] == "BUY" and position == 0:
                btc_balance = cash / price
                cash = 0
                position = 1
                self.trade.append({"action": "BUY", "price": price, "btc": btc_balance})

            elif signals[i] == "SELL" and position == 1:
                cash = btc_balance * price
                btc_balance = 0
                position = 0
                self.trade.append({"action": "SELL", "price": price, "cash": cash})

        if position == 1:
            self.final_cash = round(btc_balance * self.closes[-1], 2)
        else:
            self.final_cash = round(cash, 2)

        return self.final_cash

    def stats(self):
        total_return = ((self.final_cash - self.starting_cash) / self.starting_cash) * 100
        total_trades = len(self.trade)

        winning_trades = 0
        total_sells = 0

        for trade in self.trade:
            if trade["action"] == "SELL":
                if trade["cash"] > self.starting_cash:
                    winning_trades += 1
                total_sells += 1

        if total_sells == 0:
            win_rate = 0
        else:
            win_rate = (winning_trades / total_sells) * 100

        return {
            "total return": round(total_return, 2),
            "total trades": total_trades,
            "win rate": round(win_rate, 2),
            "final cash": self.final_cash,
        }

## 2. Find The Dataset

The CSV is stored in the same folder as this notebook, but the cell below also works if the notebook is run from the project root.

In [3]:
possible_paths = [
    Path("sample_prices.csv"),
    Path("week07-Jupyter_DataTools/notebooks/backtest/sample_prices.csv"),
]

for candidate in possible_paths:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("Could not find sample_prices.csv")

DATA_PATH

WindowsPath('sample_prices.csv')

## 3. Load The Price Data

This reads the CSV file, stores all rows, and keeps the `Close` prices ready for the strategy.

In [4]:
sim = StrategySimulator()
rows_loaded = sim.load(DATA_PATH)
rows_loaded

941

## 4. Preview The Raw Data

A quick preview helps confirm that the CSV was loaded correctly.

In [5]:
sim.data[:2]

[{'Timestamp': '1588635540.0',
  'Open': '8865.51',
  'High': '8866.87',
  'Low': '8852.85',
  'Close': '8859.89',
  'Volume': '22.10363961'},
 {'Timestamp': '1588635600.0',
  'Open': '8859.9',
  'High': '8860.73',
  'Low': '8853.1',
  'Close': '8855.68',
  'Volume': '4.8330691'}]

## 5. Check Close Prices

This project uses only the `Close` column for SMA and signal generation.

In [6]:
sim.closes[:5]

[8859.89, 8855.68, 8864.93, 8863.36, 8866.06]

## 6. Calculate SMA Values

For the first few rows there is not enough history yet, so the SMA list starts with `None` values.

In [7]:
sma_values = sim.sma(5)
sma_values[:10]

[None,
 None,
 None,
 None,
 8861.984,
 8863.546,
 8867.636,
 8869.895999999999,
 8871.045999999998,
 8872.380000000001]

## 7. Generate Trading Signals

Signals are created by comparing the current close price against the SMA value.

In [8]:
signals = sim.generate_signals(5)
signals[:15]

['No signal yet',
 'No signal yet',
 'No signal yet',
 'No signal yet',
 'BUY',
 'BUY',
 'BUY',
 'BUY',
 'SELL',
 'BUY',
 'SELL',
 'SELL',
 'SELL',
 'SELL',
 'SELL']

## 8. Run The Backtest

The backtest starts with `10000` cash and simulates all-in/all-out trades.

In [9]:
sim.backtest(5)

9837.03

## 9. Inspect A Few Trades

This shows how the simulator recorded buy and sell actions.

In [10]:
sim.trade[:6]

[{'action': 'BUY', 'price': 8866.06, 'btc': 1.1278967207530741},
 {'action': 'SELL', 'price': 8869.11, 'cash': 10003.440084998298},
 {'action': 'BUY', 'price': 8872.73, 'btc': 1.1274365482775086},
 {'action': 'SELL', 'price': 8864.26, 'cash': 9993.890697434388},
 {'action': 'BUY', 'price': 8834.69, 'btc': 1.1312101157408339},
 {'action': 'SELL', 'price': 8866.88, 'cash': 10030.304351060084}]

## 10. View Backtest Stats

These summary numbers make it easier to judge whether the strategy performed well.

In [11]:
sim.stats()

{'total return': -1.63,
 'total trades': 278,
 'win rate': 10.79,
 'final cash': 9837.03}

## 11. Compare Different SMA Periods

One nice notebook advantage is that it becomes easy to run small experiments.

In [12]:
results = []

for period in [3, 5, 10, 20]:
    sim2 = StrategySimulator()
    sim2.load(DATA_PATH)
    sim2.backtest(period)
    stats = sim2.stats()
    results.append({
        "period": period,
        "return": stats["total return"],
        "trades": stats["total trades"],
        "win rate": stats["win rate"],
    })

results

[{'period': 3, 'return': -3.12, 'trades': 386, 'win rate': 8.29},
 {'period': 5, 'return': -1.63, 'trades': 278, 'win rate': 10.79},
 {'period': 10, 'return': -2.15, 'trades': 183, 'win rate': 7.69},
 {'period': 20, 'return': -1.38, 'trades': 127, 'win rate': 15.87}]